In [2]:
import pandas as pd
import os
import zipfile


1) Realizamos una exploración inicial de los datos, mostrando la dimension, tipo de datos, numeor de valores nulos por columna y los primeros registros

In [3]:
ruta = "datasetsWC26"
archivos = os.listdir(ruta)

datasets = {}

for archivo in archivos:
    datasets[archivo] = pd.read_csv(os.path.join(ruta,archivo),sep=None,engine="python")

ParserError: field larger than field limit (131072)

In [4]:
for nombre, df in datasets.items():
    print("ARCHIVO",nombre)
    print("TAMAÑO",df.shape)
    print("COLUMNAS")
    print(df.columns)
    print("TIPOS DE DATOS:", df.dtypes)
    print("VALORES NULOS:", df.isna().sum())
    print("CABEZERA:", df.head())

ARCHIVO 2001eurocopa_fem_stats.csv
TAMAÑO (19, 47)
COLUMNAS
Index(['id_match', 'home_team', 'away_team', 'home_team_code',
       'away_team_code', 'home_score', 'away_score', 'home_penalty',
       'away_penalty', 'home_score_total', 'away_score_total', 'winner',
       'winner_reason', 'year', 'date', 'date_time', 'utc_offset_hours',
       'group_name', 'matchday_name', 'condition_humidity', 'condition_pitch',
       'condition_temperature', 'condition_weather', 'condition_wind_speed',
       'status', 'type', 'round', 'round_mode', 'match_attendance',
       'stadium_id', 'stadium_country_code', 'stadium_capacity',
       'stadium_latitude', 'stadium_longitude', 'stadium_pitch_length',
       'stadium_pitch_width', 'goals', 'penalties_missed', 'penalties',
       'red_cards', 'game_referees', 'stadium_city', 'stadium_name',
       'stadium_name_media', 'stadium_name_official', 'stadium_name_event',
       'stadium_name_sponsor'],
      dtype='object')
TIPOS DE DATOS: id_match      

2) Creamos 3 variables del tipo DataFrame

In [5]:
partidos = pd.DataFrame()
eventos = pd.DataFrame()
jugadores = pd.DataFrame()

3) Añadimos a cada DataFrame las columnas que nos piden

PARTIDOS:


In [6]:
df_mundial_202 = pd.read_csv("matches_mundial202.csv")
df_mundial_202

FileNotFoundError: [Errno 2] No such file or directory: 'matches_mundial202.csv'

In [303]:
df_mundial_2026 = pd.read_csv("matches_mundial2026.csv", sep=";")
df_mundial_2026

,match_id,date,kickoff_time_utc,stage_id,venue_id,home_team_id,away_team_id,home_score,away_score,home_penalty_score,away_penalty_score,status,result_type,home_xg,away_xg,referee_id,player_of_the_match_id,capacidad_estadio_pct,clima
0,1,11/06/2026,19:00:00,1,1,1,2,2,0,NaN,NaN,Completed,Regular,184,52,42.0,16,63.1,soleado
1,2,11/06/2026,23:00:00,1,7,3,4,2,1,NaN,NaN,Completed,Regular,145,112,5.0,58,91.2,soleado
2,3,06/12/2026,19:00:00,1,6,5,6,1,1,NaN,NaN,Completed,Regular,135,98,15.0,112,NaN,NaN
3,4,12 June 2026,23:00:00,1,3,13,14,4,1,NaN,NaN,Completed,Regular,276,88,11.0,332,88.9,soleado
4,5,"June 13, 2026",15:00:00,1,10,7,8,1,1,NaN,NaN,Completed,Regular,78,154,38.0,157,NaN,soleado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,60,26 de junio de 2026,22:00:00,1,16,22,23,1,1,NaN,NaN,Completed,Regular,112,95,8.0,557,130.5,Soleado
106,11,20260614,21:00:00,1,11,19,20,1,0,NaN,NaN,Completed,Regular,11,95,17.0,479,87.2,Nublado
107,35,20260620,21:00:00,1,14,21,23,5,1,NaN,NaN,Completed,Regular,345,88,29.0,531,74.6,DESPEJADO
108,20,2026-06-16T19:00:00Z,23:59:00,1,15,39,40,3,1,NaN,NaN,Completed,Regular,198,88,10.0,1023,61.0,Lluvia


In [304]:
partidos["fecha_partido"] = pd.concat([df_mundial_202["Date"], df_mundial_2026["date"]], ignore_index=True)
partidos["ronda"] = pd.concat([df_mundial_202["Round"], pd.Series([None]*len(df_mundial_2026))], ignore_index=True)
partidos["estadio"] = pd.concat([df_mundial_202["Venue"], pd.Series([None]*len(df_mundial_2026))], ignore_index=True)
partidos["ciudad_partido"] = partidos["estadio"].str.split(",").str[-1].str.strip()
partidos["pais_partido"] = pd.concat([df_mundial_202["Host"], pd.Series([None]*len(df_mundial_2026))], ignore_index=True)
partidos["seleccion_local"] =  pd.concat([df_mundial_202["home_team"], pd.Series([None]*len(df_mundial_2026))], ignore_index=True)
partidos["seleccion_visitante"] =  pd.concat([df_mundial_202["away_team"], pd.Series([None]*len(df_mundial_2026))], ignore_index=True)
partidos["marcador_local"] =  pd.concat([df_mundial_202["home_score"], df_mundial_2026["home_score"]], ignore_index=True)
partidos["marcador_visitante"] =  pd.concat([df_mundial_202["away_score"], df_mundial_2026["away_score"]], ignore_index=True)
partidos["prorroga"]  = pd.concat([df_mundial_202["Notes"].str.contains("extra time",case=False,na=False).map({True: "SI", False: "NO"}), pd.Series([None]*len(df_mundial_2026))], ignore_index=True)
partidos["penalty_local"] = pd.concat([df_mundial_202["home_penalty"], df_mundial_2026["home_penalty_score"]], ignore_index=True)
partidos["penalty_visitante"] = pd.concat([df_mundial_202["away_penalty"], df_mundial_2026["away_penalty_score"]], ignore_index=True)
partidos["penalty_local"] = partidos["penalty_local"].fillna(0)
partidos["penalty_visitante"] = partidos["penalty_visitante"].fillna(0)
partidos["arbitro"] =  pd.concat([df_mundial_202["Referee"], pd.Series([None]*len(df_mundial_2026))], ignore_index=True)


In [305]:
partidos

,fecha_partido,ronda,estadio,ciudad_partido,pais_partido,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,prorroga,penalty_local,penalty_visitante,arbitro
0,2022-12-18,Final,"Lusail Iconic Stadium, Lusail",Lusail,Qatar,Argentina,France,3,3,SI,4.0,2.0,Szymon Marciniak
1,2022-12-17,Third-place match,"Khalifa International Stadium, Doha",Doha,Qatar,Croatia,Morocco,2,1,NO,0.0,0.0,Abdulrahman Ibrahim Al Jassim
2,2022-12-14,Semi-finals,"Al Bayt Stadium, Al Khor",Al Khor,Qatar,France,Morocco,2,0,NO,0.0,0.0,César Arturo Ramos
3,2022-12-13,Semi-finals,"Lusail Iconic Stadium, Lusail",Lusail,Qatar,Argentina,Croatia,3,0,NO,0.0,0.0,Daniele Orsato
4,2022-12-10,Quarter-finals,"Al Thumama Stadium, ath-Thumāma",ath-Thumāma,Qatar,Morocco,Portugal,1,0,NO,0.0,0.0,Facundo Tello
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1084,26 de junio de 2026,None,None,None,None,None,None,1,1,None,0.0,0.0,None
1085,20260614,None,None,None,None,None,None,1,0,None,0.0,0.0,None
1086,20260620,None,None,None,None,None,None,5,1,None,0.0,0.0,None
1087,2026-06-16T19:00:00Z,None,None,None,None,None,None,3,1,None,0.0,0.0,None


JUGADORES:

In [306]:
df_player_stats = pd.read_csv("2026player_stats.csv",sep=";")
df_player_stats


,player_id,player_name,team_id,position,matches_played,matches_started,minutes_played,goals,assists,shots,shots_on_target,yellow_cards,red_cards,penalty_goals,own_goals,clean_sheets,saves,goals_conceded,Unnamed: 18,average_rating
0,1,JosŽ Raœl Rangel,1,GK,5,5,438,0.0,0.0,0.0,NaN,0.0,0.0,0,0,4.0,7.0,3.0,NaN,7.20
1,2,Jorge Eduardo Sanchez,1,DEF,4,4,349,0.0,1.0,1.0,NaN,1.0,0.0,0,0,NaN,NaN,NaN,NaN,6.68
2,3,CŽsar Jasib Montes,1,DEF,4,4,315,0.0,0.0,1.0,NaN,0.0,1.0,0,0,NaN,NaN,NaN,NaN,7.05
3,4,Edson Omar Alvarez,1,DEF,4,2,239,0.0,0.0,2.0,NaN,1.0,0.0,0,0,NaN,NaN,NaN,NaN,7.08
4,5,Johan Felipe Vasquez,1,DEF,4,4,360,0.0,0.0,1.0,NaN,1.0,0.0,0,0,NaN,NaN,NaN,NaN,7.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,1244,Orlando Mosquera,48,GK,3,3,270,0.0,0.0,0.0,NaN,0.0,0.0,0,0,0.0,8.0,4.0,NaN,6.80
1244,1245,Michael Amir Murillo,48,DEF,3,3,270,0.0,0.0,5.0,NaN,0.0,0.0,0,0,NaN,NaN,NaN,NaN,6.93
1245,1246,Azar’as Emmanuel Londono,48,FWD,3,0,53,0.0,0.0,1.0,NaN,0.0,0.0,0,0,NaN,NaN,NaN,NaN,6.57
1246,1247,Roderick Alonso Miller,48,DEF,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN


In [307]:
df_squads =  pd.read_csv("2026squads_and_players.csv",sep=";")
df_squads

,player_id,team_id,player_name,position,club_team,market_value_eur,caps,international_goals
0,1,1,José Raúl Rangel,GK,CD Guadalajara,8.9336,15,0
1,2,1,Jorge Eduardo Sanchez,DEF,PAOK Saloniki,24.6286,59,3
2,3,1,César Jasib Montes,DEF,FC Lokomotiv Moscow,10.9126,69,4
3,4,1,Edson Omar Alvarez,DEF,Fenerbahçe SK,10.8759,100,7
4,5,1,Johan Felipe Vasquez,DEF,Genoa CFC,18.2654,47,3
...,...,...,...,...,...,...,...,...
1243,1244,48,Orlando Mosquera,GK,Al Fayha FC,0.7000,50,0
1244,1245,48,Michael Amir Murillo,DEF,Be ş ikta ş JK,7.8302,94,9
1245,1246,48,Azarías Emmanuel Londono,FWD,CD Universidad Católica,4.4904,13,1
1246,1247,48,Roderick Alonso Miller,DEF,Turan Tovuz,NaN,49,2


In [308]:
df_teams = pd.read_csv("2026nationalteams.csv")

In [309]:
jugadores["nombre_jugador"] = df_player_stats["player_name"]
jugadores["team_id"] = df_player_stats["team_id"]
jugadores = pd.merge(jugadores,df_teams[["team_id","team_name"]].drop_duplicates(subset="team_id"), on="team_id",how="left")
jugadores = jugadores.drop(columns=["team_id"])
jugadores = jugadores.rename(columns={"team_name": "seleccion_nacional"})
jugadores["club_jugador"] = df_squads["club_team"]
#jugadores["partidos_seleccion"] = 
jugadores["posicion"] = df_player_stats["position"]
jugadores["valor_mercado"] = df_squads["market_value_eur"]
jugadores["partidos_jugados"] = df_player_stats["matches_played"]
jugadores["minutos_jugados"] = df_player_stats["minutes_played"]
jugadores["goles"] = df_player_stats["goals"]
jugadores["asistencias"] = df_player_stats["assists"]
jugadores["tiros"] = df_player_stats["shots"]
jugadores["tarjetas_amarillas"] = df_player_stats["yellow_cards"]
jugadores["tarjetas_rojas"] = df_player_stats["red_cards"]
jugadores["paradas"] = df_player_stats["saves"]
jugadores["paradas"] = jugadores["paradas"].fillna(0) ## cambiamos los valores NaN por 0 ya que la mayoria de jugadores no son porteros y pues, no hacen ninguna parada
jugadores["goles_concedidos"] = df_player_stats["goals_conceded"]
jugadores["calificacion_media"] = df_player_stats["average_rating"]

In [310]:
jugadores

,nombre_jugador,seleccion_nacional,club_jugador,posicion,valor_mercado,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media
0,JosŽ Raœl Rangel,Mexico,CD Guadalajara,GK,8.9336,5,438,0.0,0.0,0.0,0.0,0.0,7.0,3.0,7.20
1,Jorge Eduardo Sanchez,Mexico,PAOK Saloniki,DEF,24.6286,4,349,0.0,1.0,1.0,1.0,0.0,0.0,NaN,6.68
2,CŽsar Jasib Montes,Mexico,FC Lokomotiv Moscow,DEF,10.9126,4,315,0.0,0.0,1.0,0.0,1.0,0.0,NaN,7.05
3,Edson Omar Alvarez,Mexico,Fenerbahçe SK,DEF,10.8759,4,239,0.0,0.0,2.0,1.0,0.0,0.0,NaN,7.08
4,Johan Felipe Vasquez,Mexico,Genoa CFC,DEF,18.2654,4,360,0.0,0.0,1.0,1.0,0.0,0.0,NaN,7.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,Orlando Mosquera,Panama,Al Fayha FC,GK,0.7000,3,270,0.0,0.0,0.0,0.0,0.0,8.0,4.0,6.80
1244,Michael Amir Murillo,Panama,Be ş ikta ş JK,DEF,7.8302,3,270,0.0,0.0,5.0,0.0,0.0,0.0,NaN,6.93
1245,Azar’as Emmanuel Londono,Panama,CD Universidad Católica,FWD,4.4904,3,53,0.0,0.0,1.0,0.0,0.0,0.0,NaN,6.57
1246,Roderick Alonso Miller,Panama,Turan Tovuz,DEF,NaN,0,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN


EVENTOS: 

In [311]:
df_events = pd.read_csv("2026match_register.csv", sep=";")

In [312]:
df_events

,event_id,match_id,minute,event_type,team_id,player_id
0,1,1.0,9.0,Goal,1.0,16.0
1,2,1.0,9.0,Assist,1.0,6.0
2,3,1.0,17.0,Yellow Card,2.0,30.0
3,4,1.0,23.0,Yellow Card,1.0,26.0
4,5,1.0,55.0,Red Card,2.0,39.0
...,...,...,...,...,...,...
859,41,4.0,301.0,Assist,13.0,328.0
860,206,27.0,350.0,Goal,5.0,182.0
861,138,18.0,420.0,Goal,36.0,919.0
862,698,91.0,500.0,Goal,36.0,919.0


In [313]:
eventos["minuto_evento"] = df_events["minute"]
eventos["index_partido"] = df_events["match_id"]
eventos["player_id"] = df_events["player_id"]
eventos = pd.merge(eventos, df_player_stats[["player_id","player_name","team_id"]], on="player_id", how="left")
eventos["tipo_evento"] = df_events["event_type"]
eventos = pd.merge(eventos, df_teams[["team_id","team_name"]].drop_duplicates(subset="team_id"), on="team_id", how="left")
eventos = eventos.rename(columns = {"player_name":"nombre_jugador", "team_name":"seleccion_evento"})
eventos = eventos.drop(columns=["player_id", "team_id"])

In [314]:
eventos

,minuto_evento,index_partido,nombre_jugador,tipo_evento,seleccion_evento
0,9.0,1.0,Juli‡n AndrŽs Quinones,Goal,Mexico
1,9.0,1.0,Erik Antonio Lira,Assist,Mexico
2,17.0,1.0,Teboho Mokoena,Yellow Card,South Africa
3,23.0,1.0,Brian Gutierrez,Yellow Card,Mexico
4,55.0,1.0,Sphephelo S'miso Sithole,Red Card,South Africa
...,...,...,...,...,...
859,301.0,4.0,Alexander Michael Freeman,Assist,USA
860,350.0,27.0,Naceur Mohamed Manai,Goal,Qatar
861,420.0,18.0,Erling Braut Haaland,Goal,Norway
862,500.0,91.0,Erling Braut Haaland,Goal,Norway


Reescribimos la fecha de los partidos con la misma estructura

In [1]:
# Formato de año-mes-dia
estructura1 = partidos["fecha_partido"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}")
partidos.loc[estructura1,"fecha_partido"] = pd.to_datetime(partidos.loc[estructura1,"fecha_partido"],format="%Y-%m-%d",errors="coerce")

# Formato se añomesdia
estructura2 = partidos["fecha_partido"].astype(str).str.fullmatch(r"\d{8}")
partidos.loc[estructura2,"fecha_partido"] = pd.to_datetime(partidos.loc[estructura2,"fecha_partido"],format="%Y%m%d",errors="coerce")

# Formato año-mes-dia:horas:segundos
estructura3 = partidos["fecha_partido"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z")
partidos.loc[estructura3,"fecha_partido"] = pd.to_datetime(partidos.loc[estructura3,"fecha_partido"],format="%Y-%m-%dT%H:%M:%SZ",errors="coerce")

# Formato dia mes(escrito) año
estructura4 = partidos["fecha_partido"].astype(str).str.fullmatch(r"\d{1,2} de [a-zA-Záéíóú]+ de \d{4}")
partidos.loc[estructura4,"fecha_partido"] = partidos.loc[estructura4, "fecha_partido"].str.replace(" de enero de ","-01-", regex=False).str.replace(" de febrero de ","-02-", regex=False).str.replace(" de marzo de ","-03-", regex=False).str.replace(" de abril de ","-04-", regex=False).str.replace(" de mayo de ","-05-", regex=False).str.replace(" de junio de ","-06-", regex=False).str.replace(" de julio de ","-07-", regex=False).str.replace(" de agosto de ","-08-", regex=False).str.replace(" de septiembre de ","-09-", regex=False).str.replace(" de octubre de ","-10-", regex=False).str.replace(" de noviembre de ","-11-", regex=False).str.replace(" de diciembre de ","-12-", regex=False)
partidos.loc[estructura4, "fecha_partido"] = pd.to_datetime(partidos.loc[estructura4, "fecha_partido"],format="%d-%m-%Y",errors="coerce")

partidos["fecha_partido"] = pd.to_datetime(partidos["fecha_partido"],errors="coerce").dt.strftime("%Y-%m-%d")

partidos


NameError: name 'partidos' is not defined